In [ ]:
import geopandas as gpd
import pandas as pd
import dask.dataframe as dd
import numpy as np
from IPython.display import display
import os
import xarray as xr
from xarray import open_dataset
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')


Investigate Cloud Optimised data (saved part csv files in the subfolder named `cloud_optimised_download`)

In [2]:
co_folder_path = Path("cloud_optimised_download")

pattern = str(co_folder_path / "*.csv")
ddf = dd.read_csv(pattern)
co_data = ddf.compute()

In [3]:
co_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15210 entries, 0 to 15209
Data columns (total 51 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   filename          15210 non-null  object 
 1   site              15210 non-null  object 
 2   TIME              15210 non-null  object 
 3   LATITUDE          15210 non-null  float64
 4   LONGITUDE         15210 non-null  float64
 5   PL_CMP            15210 non-null  float64
 6   PL_CRS            15210 non-null  float64
 7   PL_SPD            15210 non-null  float64
 8   WDIR              15210 non-null  float64
 9   WSPD              15210 non-null  float64
 10  WIND_H            15210 non-null  float64
 11  WIND_FLAG         15210 non-null  int64  
 12  ATMP              15210 non-null  float64
 13  ATMP_H            15210 non-null  float64
 14  ATMP_FLAG         15210 non-null  int64  
 15  AIRT              15210 non-null  float64
 16  AIRT_H            15210 non-null  float6

Investigate WFS download data (renamed the downloaded csv file to `wfs_download.csv`)

In [4]:
# read data
wfs_data = gpd.read_file("wfs_download.csv")
wfs_data.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   FID                  14 non-null     object  
 1   file_id              14 non-null     object  
 2   url                  14 non-null     object  
 3   size                 14 non-null     object  
 4   time_coverage_start  14 non-null     object  
 5   time_coverage_end    14 non-null     object  
 6   colour               14 non-null     object  
 7   geom                 14 non-null     object  
 8   platform_code        14 non-null     object  
 9   cruise_id            14 non-null     object  
 10  vessel_name          14 non-null     object  
 11  geometry             0 non-null      geometry
dtypes: geometry(1), object(11)
memory usage: 1.4+ KB


Evaluation Metric: File Name (parsed file name from `wfs_data.url` vs `co_data.filename`)

In [24]:
def parse_filename(url):
    """Parse the filename from the URL like IMOS/SOOP/SOOP-ASF/ZMFR_Tangaroa/flux_product/2018/IMOS_SOOP-ASF_FMT_20180108T000200Z_ZMFR_FV02.nc"""
    return url.split("/")[-1].strip()
wfs_data['parsed_filename'] = wfs_data['url'].apply(parse_filename)
unique_filenames = co_data['filename'].unique()
wfs_data['filename_in_co'] = wfs_data['parsed_filename'].apply(lambda x: x in unique_filenames)
wfs_data['filename_in_co'].value_counts()
wfs_data['parsed_filename'].unique().size

print("Unique filenames in CO data:", unique_filenames.size)
print("Unique filenames in WFS data:", wfs_data['parsed_filename'].unique().size)
# Summary of filename matching between WFS data and CO data
display(wfs_data['filename_in_co'].value_counts())

Unique filenames in CO data: 14
Unique filenames in WFS data: 14


filename_in_co
True    14
Name: count, dtype: int64

Evaluation Metric: Data Volumne (fetch data from thredds through `wfs_data.url` vs data in `co_data` with the same filename)

In [25]:
thredds_url = "https://thredds.aodn.org.au/thredds/dodsC/"

# use the first url listed in wfs_data
index_of_file = 2
thredds_file_url = wfs_data["url"].loc[index_of_file]
nc = xr.open_dataset(thredds_url + thredds_file_url)

# use the subset of co_data with the same filename
filename_to_match = wfs_data["parsed_filename"].loc[index_of_file]
print("Filename to match:", filename_to_match)
co_data_subset = co_data[co_data["filename"] == filename_to_match]

# Display NetCDF variables and attributes vs CO data columns
print("\nNetCDF Variables and Attributes:")
print(list(nc.variables.keys()))

# Display CO data columns
print("\nCO Data Columns:")
print(co_data_subset.columns.tolist())

# Check how many NetCDF variables match CO data columns
matching_vars = [var for var in nc.variables.keys() if var in co_data_subset.columns]
print(f"\nMatching Variables for NetCDF in CO Data: {len(matching_vars)}/{len(nc.variables.keys())}")

Filename to match: IMOS_SOOP-ASF_FMT_20221016T020800Z_VLMJ_FV02.nc

NetCDF Variables and Attributes:
['TIME', 'LATITUDE', 'LONGITUDE', 'PL_CMP', 'PL_CRS', 'PL_SPD', 'WDIR', 'WSPD', 'WIND_H', 'WIND_FLAG', 'ATMP', 'ATMP_H', 'ATMP_FLAG', 'AIRT', 'AIRT_H', 'AIRT_FLAG', 'RELH', 'RELH_H', 'RELH_FLAG', 'TEMP', 'TEMP_H', 'TEMP_FLAG', 'RAIN_AMOUNT', 'RAIN_AMOUNT_H', 'RAIN_AMOUNT_FLAG', 'SW', 'SW_H', 'SW_FLAG', 'LW', 'LW_H', 'LW_FLAG', 'HS', 'HL', 'H_RAIN', 'TAU', 'SST', 'HEAT_NET', 'MASS_NET', 'LW_NET', 'SW_NET', 'WSPD10M', 'AIRT1_5M', 'AIRT2_0M', 'RELH1_5M', 'RELH2_0M', 'history']

CO Data Columns:
['filename', 'site', 'TIME', 'LATITUDE', 'LONGITUDE', 'PL_CMP', 'PL_CRS', 'PL_SPD', 'WDIR', 'WSPD', 'WIND_H', 'WIND_FLAG', 'ATMP', 'ATMP_H', 'ATMP_FLAG', 'AIRT', 'AIRT_H', 'AIRT_FLAG', 'RELH', 'RELH_H', 'RELH_FLAG', 'TEMP', 'TEMP_H', 'TEMP_FLAG', 'RAIN_AMOUNT', 'RAIN_AMOUNT_H', 'RAIN_AMOUNT_FLAG', 'SW', 'SW_H', 'SW_FLAG', 'LW', 'LW_H', 'LW_FLAG', 'HS', 'HL', 'H_RAIN', 'TAU', 'SST', 'HEAT_NET', 'MASS

In [26]:
# Compare data volume
nc_dims = dict(nc.dims)
# Flatten the dimensions to get total number of entries
nc_total_entries = np.prod(list(nc.dims.values()))
print("Data volume in THREDDS file:", nc_total_entries)
print("Data volume in CO data subset:", co_data_subset.shape[0])

Data volume in THREDDS file: 1185
Data volume in CO data subset: 1185


In [27]:
# Compare time coverage
nc_time = nc['TIME']
nc_time_values = nc_time[:]
nc_time_start = nc_time_values.min()
nc_time_end = nc_time_values.max()
co_time_values = pd.to_datetime(co_data_subset['TIME'].values)
co_time_start = co_time_values.min()
co_time_end = co_time_values.max()
print(f"THREDDS time coverage: {nc_time_start} to {nc_time_end}")
print(f"CO data time coverage: {co_time_start} to {co_time_end}")

THREDDS time coverage: <xarray.DataArray 'TIME' ()> Size: 8B
array('2022-10-16T02:07:59.999984896', dtype='datetime64[ns]') to <xarray.DataArray 'TIME' ()> Size: 8B
array('2022-10-16T23:56:00.000014336', dtype='datetime64[ns]')
CO data time coverage: 2022-10-16 02:07:59.999984896 to 2022-10-16 23:56:00.000014336


In [28]:
# Compare spatial coverage
nc_lat = nc['LATITUDE'][:]
nc_lon = nc['LONGITUDE'][:]
nc_lat_min, nc_lat_max = nc_lat.min(), nc_lat.max()
nc_lon_min, nc_lon_max = nc_lon.min(), nc_lon.max()

co_lat = co_data_subset['LATITUDE'].values
co_lon = co_data_subset['LONGITUDE'].values
co_lat_min, co_lat_max = co_lat.min(), co_lat.max()
co_lon_min, co_lon_max = co_lon.min(), co_lon.max()
print(f"THREDDS spatial coverage: LATITUDE {nc_lat_min} to {nc_lat_max}, LONGITUDE {nc_lon_min} to {nc_lon_max}")
print(f"CO data spatial coverage: LATITUDE {co_lat_min} to {co_lat_max}, LONGITUDE {co_lon_min} to {co_lon_max}")

THREDDS spatial coverage: LATITUDE <xarray.DataArray 'LATITUDE' ()> Size: 4B
array(-12.3257, dtype=float32) to <xarray.DataArray 'LATITUDE' ()> Size: 4B
array(-12.0445, dtype=float32), LONGITUDE <xarray.DataArray 'LONGITUDE' ()> Size: 4B
array(96.762, dtype=float32) to <xarray.DataArray 'LONGITUDE' ()> Size: 4B
array(97.038, dtype=float32)
CO data spatial coverage: LATITUDE -12.32569980621338 to -12.044500350952148, LONGITUDE 96.76200103759766 to 97.03800201416016


In [34]:
# On a specific variable, compare in two datasets, at a specific timestamp
variable_to_compare = 'TEMP'
specific_time = pd.Timestamp(nc_time_values[0].values)
nc_var_data = nc[variable_to_compare].sel(TIME=nc_time_values[0]).values.flatten()
co_data_subset['TIME'] = pd.to_datetime(co_data_subset['TIME'])
co_var_data = co_data_subset[co_data_subset['TIME'] == specific_time][variable_to_compare].values

print(f"Comparison for {variable_to_compare} at {specific_time}:")
print(f"NetCDF: {len(nc_var_data)} values")
print(f"CO data: {len(co_var_data)} values")
print(f"Values match: {np.allclose(nc_var_data, co_var_data, rtol=1e-5)}")

Comparison for TEMP at 2022-10-16 02:07:59.999984896:
NetCDF: 1 values
CO data: 15 values
Values match: True
